# 🛡️ Tesseract AI: Google Colab Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manideep1428/razorpay/blob/main/colab_train.ipynb)

Train **GraphSAGE GNN** (Signup Trust Model) and **LightGBM** (Payment Abuse Model) directly from the 10M dataset hosted on Hugging Face (`vicky1428/fraudshield-10m`) using GPU acceleration.

---
### Models Trained:
1. **Signup Trust Model (`SignupGraphSAGE`)**: Predicts user trust score (0-100), risk score, and 4-tier signup action (`ALLOW`, `ALLOW_FLAG_REVIEW`, `ALLOW_HIGH_PRIORITY_REVIEW`, `TEMP_SUSPEND_MANUAL_REVIEW`).
2. **Payment Abuse Model (`PaymentAbuseModel`)**: Multi-class LightGBM (`0=legit`, `1=trial_abuse`, `2=discount_abuse`, `3=payment_fraud`) with tiered decisioning (`ALLOW`, `ALLOW_FLAG_REVIEW`, `ALLOW_HIGH_PRIORITY_REVIEW`, `BLOCK`).

## 1. Check GPU Compute Device
Colab provides free T4 GPUs. If GPU is not enabled, go to **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Clone Repository & Install Missing Dependencies
Google Colab already comes pre-installed with PyTorch (CUDA), pandas, numpy, and scikit-learn. We only install the additional required libraries and install `tesseract` package in editable mode.

In [ ]:
import os
import sys

# 1. Clone repository to /content/razorpay (avoids directory nesting)
if not os.path.exists("/content/razorpay"):
    !git clone https://github.com/manideep1428/razorpay.git /content/razorpay

%cd /content/razorpay
!git pull origin main

# 2. Add src to Python path
if "/content/razorpay/src" not in sys.path:
    sys.path.insert(0, "/content/razorpay/src")

# 3. Install only missing libraries & install package
!pip install -q torch-geometric lightgbm datasets huggingface-hub pyarrow
!pip install -q -e .

## 3. (Optional) Mount Google Drive to Persist Checkpoints
Uncomment the drive mount lines below if you want trained checkpoints permanently saved to your Google Drive.

In [ ]:
import os

# To save to Google Drive, uncomment these 3 lines:
# from google.colab import drive
# drive.mount('/content/drive')
# SAVE_DIR = '/content/drive/MyDrive/fraudshield_models'

# Default: save to repo artifacts folder
SAVE_DIR = '/content/razorpay/artifacts'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models will be saved to: {SAVE_DIR}")

## 4. Configure Training Parameters

Choose your training scale:
* **Micro test**: `MAX_ROWS = 100` (~3 seconds)
* **Fast verification**: `MAX_ROWS = 50_000` (~1 minute)
* **Standard benchmark**: `MAX_ROWS = 500_000` (~4 minutes on T4 GPU)
* **Large scale**: `MAX_ROWS = 2_000_000` (~15 minutes on T4 GPU)
* **Full 10M dataset**: Set `MAX_ROWS = None` (High-RAM runtime recommended)

In [ ]:
import torch

REPO_ID = "vicky1428/fraudshield-10m"
MAX_ROWS = 500_000        # Choose: 100, 50000, 500000, 2000000, or None for all
GNN_EPOCHS = 100          # GraphSAGE epochs
LIGHTGBM_TREES = 300      # LightGBM boosting trees
CALIBRATE = True          # Enable probability calibration

# Auto-detect GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Compute device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU Name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Note: GPU not detected. Using CPU. Switch to T4 GPU via Runtime > Change runtime type.")

max_rows_flag = f"--max-rows {MAX_ROWS}" if MAX_ROWS else ""
calib_flag = "--calibrate" if CALIBRATE else ""

cmd = f"PYTHONPATH=src python train.py --repo-id {REPO_ID} {max_rows_flag} --epochs {GNN_EPOCHS} --trees {LIGHTGBM_TREES} --device {DEVICE} --save-dir {SAVE_DIR} {calib_flag}"
print("\nReady to run:", cmd)

## 5. Execute Training on Hugging Face Dataset
Smart shard loading will automatically download only the needed shards.

In [ ]:
!{cmd}

## 6. Evaluate Models on Held-Out Test Split from Hugging Face

In [ ]:
!PYTHONPATH=src python test.py --repo-id {REPO_ID} --test-rows 10000 --artifacts-dir {SAVE_DIR}

## 7. Interactive Live Inference Demo
Run sample predictions with both trained models to verify decisioning.

In [ ]:
import os
import sys
if "/content/razorpay/src" not in sys.path:
    sys.path.insert(0, "/content/razorpay/src")

import pandas as pd
from tesseract.inference.predict_signup import SignupPredictor
from tesseract.inference.predict_payment import PaymentPredictor
from tesseract.config import FeatureConfig
from tesseract.utils.synthetic import (
    synthesize_payment_dataset,
    synthesize_signup_dataset,
    synthesize_signup_edges,
)
from tesseract.utils.preprocessing import build_graph_data

# 1. Test Payment Abuse Model
pay_path = os.path.join(SAVE_DIR, "payment_abuse_lgbm.joblib")
pay_predictor = PaymentPredictor(pay_path)
sample_txs = synthesize_payment_dataset(n=5, seed=123)
results = pay_predictor.score_batch(sample_txs)

display_cols = ['payment_risk_score', 'abuse_type', 'risk_level', 'decision']
print("=" * 60)
print("PAYMENT ABUSE PREDICTIONS:")
print("=" * 60)
print(results[display_cols].to_string(index=False))

# 2. Test Signup Trust GNN Model
signup_path = os.path.join(SAVE_DIR, "signup_graphsage.pt")
signup_predictor = SignupPredictor(signup_path)
sample_signups = synthesize_signup_dataset(n=10, seed=123)
feat_cfg = FeatureConfig()
nodes = sample_signups[feat_cfg.signup_numeric_features]
edges = synthesize_signup_edges(len(sample_signups), avg_degree=4.0, seed=123)
graph = build_graph_data(nodes, edges, labels=sample_signups["label"])

signup_res = signup_predictor.predict_single_node(node_idx=0, data=graph)
print("\n" + "=" * 60)
print("SIGNUP TRUST PREDICTION (Single Node #0):")
print("=" * 60)
for k in ['trust_score', 'risk_score', 'risk_level', 'decision']:
    print(f"  {k:<12}: {signup_res[k]}")

print("\n[SUCCESS] Both model predictors loaded and verified in Google Colab!")

## 8. (Optional) Upload Trained Models to Hugging Face Hub
Save your trained weights permanently to Hugging Face.

In [ ]:
# from huggingface_hub import HfApi, create_repo
# HF_MODEL_REPO = "your-username/fraudshield-models"
# HF_TOKEN = "hf_..."  # Your HF Write Token
# api = HfApi(token=HF_TOKEN)
# create_repo(HF_MODEL_REPO, repo_type="model", token=HF_TOKEN, exist_ok=True)
# api.upload_file(path_or_fileobj=f"{SAVE_DIR}/signup_graphsage.pt", path_in_repo="signup_graphsage.pt", repo_id=HF_MODEL_REPO, repo_type="model")
# api.upload_file(path_or_fileobj=f"{SAVE_DIR}/payment_abuse_lgbm.joblib", path_in_repo="payment_abuse_lgbm.joblib", repo_id=HF_MODEL_REPO, repo_type="model")
# print(f"Trained models uploaded to https://huggingface.co/{HF_MODEL_REPO}")